# Select images for analysis

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import sys
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d00_utils import dirnames as dn
from src.d00_utils import utilities as utils

In [ ]:
input_dirpath = Path(input())

In [ ]:
imgnames = [path.name for path in input_dirpath.glob('*.ome.tif')]
imgnames.sort()
df = pd.DataFrame({'image name': imgnames})

In [ ]:
df.to_csv(input_dirpath / 'notes.csv')

In [ ]:
wellcond_df_path = Path(input())

In [ ]:
wellcond_df = pd.read_csv(wellcond_df_path)
wellcond_df.head()

In [ ]:
num_rows_premerge = len(df)
#img_list_df = pd.merge(df, wellcond_df, how='left', on=['wellID'], suffixes=['', '_y'], validate='many_to_one')
df = pd.merge(df, wellcond_df, how='left', on=['experiment', 'wellID'], suffixes=['', '_y'], validate='many_to_one')
num_rows_postmerge = len(df)
assert num_rows_premerge==num_rows_postmerge
df

In [ ]:
tables_dirpath = utils.get_proc_dirpath(input_dirpath) / dn.tables_dirname
tablename = 'img_counts.csv'
counts = df.groupby(['wellID'])['image name'].count()
counts = counts.rename('num imgs')
counts

In [ ]:
counts.to_csv(tables_dirpath / tablename, index=False)

In [ ]:
#experiment = ['CE033A']
tx_to_analyze = ['281', '283']

In [ ]:
#selected_df = df.loc[df['experiment']==experiment]
conds = [(df['tx']==tx) for tx in tx_to_analyze]

selection = conds[0]
for i in range(1, len(conds)):
    selection = selection | conds[i]

selected_df = df[selection]
selected_df

In [ ]:
#num_selected_imgs = 20
#selected_df = selected_df.groupby('wellID').sample(n=num_selected_imgs, random_state=1)
#selected_df

In [ ]:
tablename = 'selected_imgs.csv'
utils.safe_save_csv(selected_df, tables_dirpath / tablename)